# Phase 4: Feature Engineering (RFM Analysis)

This notebook transforms transactional data into meaningful customer-level features for clustering.

**Goals:**
- Compute RFM (Recency, Frequency, Monetary) metrics
- Normalize values for clustering
- Prepare features for K-Means
- Save RFM dataset
- Save scaled dataset

**Tracking Milestone:**
✔ Compute RFM (Recency, Frequency, Monetary)
✔ Normalize values for clustering
✔ Prepare features for K-Means
✔ Save RFM dataset
✔ Save scaled dataset

In [1]:
# Cell 1: Imports & Load Data

import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/processed/online_retail_cleaned.csv")

print("✓ Data loaded successfully")
print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

✓ Data loaded successfully
Shape: (5000, 9)

First 5 rows:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Amount
0,INV004989,PROD0026,product prod0026,30,2011-04-16,17.859765,12050,United Kingdom,535.792956
1,INV003926,PROD0044,product prod0044,26,2011-11-13,11.229507,12487,Sweden,291.967174
2,INV001944,PROD0003,product prod0003,46,2011-08-25,29.482945,12070,Netherlands,1356.215475
3,INV008629,PROD0085,product prod0085,19,2011-07-07,17.730730,12479,Germany,336.883862
4,INV005391,PROD0089,product prod0089,21,2011-01-26,33.040955,12047,Netherlands,693.860065


In [2]:
# Cell 2: Convert InvoiceDate + Define Reference Date

# We calculate recency based on the last day of transactions
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Reference date is one day after the last purchase
reference_date = df["InvoiceDate"].max() + timedelta(days=1)

print(f"Last transaction date: {df['InvoiceDate'].max().date()}")
print(f"Reference date for recency: {reference_date.date()}")

Last transaction date: 2011-11-30
Reference date for recency: 2011-12-01


In [3]:
# Cell 3: Calculate RFM Metrics

# Recency: days since last purchase
rfm = df.groupby("CustomerID").agg({
    "InvoiceDate": lambda x: (reference_date - x.max()).days,
    "InvoiceNo": "nunique",      # Frequency = number of invoices
    "Amount": "sum"               # Monetary = total spent
})

rfm.columns = ["Recency", "Frequency", "Monetary"]

print("✓ RFM metrics calculated")
print(f"\nShape: {rfm.shape}")
print(f"\nFirst 5 rows:")
rfm.head()

✓ RFM metrics calculated

Shape: (500, 3)

First 5 rows:


,Recency,Frequency,Monetary
CustomerID,,,
12000,1,7,4120.286292
12001,12,12,9248.846965
12002,144,6,3084.418018
12003,125,9,2900.167999
12004,140,4,1904.703400


In [4]:
# Cell 4: Check Basic Statistics

print("="*80)
print("RFM STATISTICS")
print("="*80)

print("\nRecency (days since last purchase):")
print(rfm["Recency"].describe())

print("\nFrequency (number of purchases):")
print(rfm["Frequency"].describe())

print("\nMonetary (total spend in £):")
print(rfm["Monetary"].describe())

RFM STATISTICS

Recency (days since last purchase):
count    500.00000
mean      38.48200
std       39.42642
min        1.00000
25%       11.00000
50%       25.00000
75%       49.25000
max      225.00000
Name: Recency, dtype: float64

Frequency (number of purchases):
count    500.000000
mean       9.992000
std        3.072265
min        3.000000
25%        8.000000
50%       10.000000
75%       12.000000
max       21.000000
Name: Frequency, dtype: float64

Monetary (total spend in £):
count      500.000000
mean      6313.523606
std       2686.775480
min        748.485856
25%       4223.047720
50%       6239.259385
75%       7992.639951
max      14402.406803
Name: Monetary, dtype: float64


In [5]:
# Cell 5: Handle Outliers / Skew (Log Transform Monetary)

# The Monetary metric is usually highly skewed → log transform improves clustering
rfm["Monetary_log"] = np.log1p(rfm["Monetary"])

print("✓ Log transformation applied to Monetary")
print(f"\nMonetary (original):")
print(rfm["Monetary"].describe())

print(f"\nMonetary_log (transformed):")
print(rfm["Monetary_log"].describe())

print(f"\nFirst 5 rows:")
rfm.head()

✓ Log transformation applied to Monetary

Monetary (original):
count      500.000000
mean      6313.523606
std       2686.775480
min        748.485856
25%       4223.047720
50%       6239.259385
75%       7992.639951
max      14402.406803
Name: Monetary, dtype: float64

Monetary_log (transformed):
count    500.000000
mean       8.643936
std        0.492847
min        6.619387
25%        8.348549
50%        8.738777
75%        8.986401
max        9.575220
Name: Monetary_log, dtype: float64

First 5 rows:


,Recency,Frequency,Monetary,Monetary_log
CustomerID,,,,
12000,1,7,4120.286292,8.323921
12001,12,12,9248.846965,9.132362
12002,144,6,3084.418018,8.034442
12003,125,9,2900.167999,7.972869
12004,140,4,1904.703400,7.552606


In [6]:
# Cell 6: Select Features for Clustering

# We'll use:
# - Recency
# - Frequency
# - Monetary_log

features = rfm[["Recency", "Frequency", "Monetary_log"]]

print("✓ Features selected for clustering")
print(f"\nShape: {features.shape}")
print(f"\nColumns: {list(features.columns)}")
print(f"\nFirst 5 rows:")
features.head()

✓ Features selected for clustering

Shape: (500, 3)

Columns: ['Recency', 'Frequency', 'Monetary_log']

First 5 rows:


,Recency,Frequency,Monetary_log
CustomerID,,,
12000,1,7,8.323921
12001,12,12,9.132362
12002,144,6,8.034442
12003,125,9,7.972869
12004,140,4,7.552606


In [7]:
# Cell 7: Scaling Features (Important for K-Means)

# K-Means is sensitive to feature scales → we standardize all features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

scaled_df = pd.DataFrame(
    scaled_features,
    columns=["Recency_scaled", "Frequency_scaled", "Monetary_scaled"],
    index=rfm.index
)

print("✓ Features scaled using StandardScaler")
print(f"\nShape: {scaled_df.shape}")
print(f"\nMean (should be ~0):")
print(scaled_df.mean())
print(f"\nStd Dev (should be ~1):")
print(scaled_df.std())
print(f"\nFirst 5 rows:")
scaled_df.head()

✓ Features scaled using StandardScaler

Shape: (500, 3)

Mean (should be ~0):
Recency_scaled      1.776357e-17
Frequency_scaled   -2.557954e-16
Monetary_scaled     1.543654e-15
dtype: float64

Std Dev (should be ~1):
Recency_scaled      1.001002
Frequency_scaled    1.001002
Monetary_scaled     1.001002
dtype: float64

First 5 rows:


,Recency_scaled,Frequency_scaled,Monetary_scaled
CustomerID,,,
12000,-0.951634,-0.974850,-0.649971
12001,-0.672354,0.654244,0.992023
12002,2.679008,-1.300668,-1.237919
12003,2.196615,-0.323212,-1.362979
12004,2.577451,-1.952306,-2.216558


In [8]:
# Cell 8: Save Engineered Datasets

rfm.to_csv("../data/processed/rfm_raw.csv")
scaled_df.to_csv("../data/processed/rfm_scaled.csv")

print("="*80)
print("PHASE 4 - FEATURE ENGINEERING COMPLETED")
print("="*80)
print(f"\n✓ RFM features saved to: data/processed/rfm_raw.csv")
print(f"✓ Scaled dataset saved to: data/processed/rfm_scaled.csv")
print(f"\nDatasets:")
print(f"  • RFM Raw: {rfm.shape[0]} customers × {rfm.shape[1]} features")
print(f"  • RFM Scaled: {scaled_df.shape[0]} customers × {scaled_df.shape[1]} features")
print(f"\n✓ Ready for Phase 5: Clustering (K-Means)")
print("="*80)

PHASE 4 - FEATURE ENGINEERING COMPLETED

✓ RFM features saved to: data/processed/rfm_raw.csv
✓ Scaled dataset saved to: data/processed/rfm_scaled.csv

Datasets:
  • RFM Raw: 500 customers × 4 features
  • RFM Scaled: 500 customers × 3 features

✓ Ready for Phase 5: Clustering (K-Means)
